# Visão Computacional com CNNs e Transformers
## 🎓 Faculdade Infnet — Pós-Graduação em Inteligência Artificial & Machine Learning
### Aula 4: Vision Transformers (ViT) com a Biblioteca `transformers` da Hugging Face

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/allanspadini/curso-vision-transformers-infnet/blob/main/aula_04_vision_transformers/aula_04_vision_transformers.ipynb)

---

### 🎯 Metodologia Pedagógica: Situação-Problema ➔ Solução de Engenharia ➔ Prática com Hugging Face

Neste laboratório prático, utilizamos o ecossistema padrão da indústria para modelos fundacionais de Visão: a biblioteca **`transformers` da Hugging Face** em conjunto com **PyTorch**, aplicando os conceitos da apresentação da Aula 04.

**Roteiro do Laboratório:**
1. **Setup de Hardware & GPU**: Diagnóstico de aceleração CUDA e sementes determinísticas.
2. **Situação-Problema Agritech**: Diagnóstico fitopatológico em folhas de feijoeiro com o dataset `AI-Lab-Makerere/beans` (3 classes: *Angular Leaf Spot*, *Bean Rust* e *Healthy*).
3. **Inferência Zero-Shot de Alto Nível com `transformers.pipeline`**: Como o ViT original pré-treinado no ImageNet-1k se comporta off-the-shelf antes do fine-tuning.
4. **Anatomia Tensorial do ViT na Hugging Face**: Patch Embeddings ($16 \times 16$), Token `[CLS]`, Position Embeddings ($197 \times 768$) e Encoder Pre-LN.
5. **Pré-Processamento com `ViTImageProcessor`**: Redimensionamento, normalização ImageNet e transformações funcionais via `datasets.with_transform`.
6. **Regularização de Dados (CutMix)**: Mitigando o gargalo de *Data Hunger* em datasets de escala moderada (~1000 amostras).
7. **Fine-Tuning com `Trainer` e `TrainingArguments`**: Treinamento supervisionado com a API canônica da Hugging Face, decaimento de peso e agendador cossenoidal.
8. **Avaliação com `Trainer.predict()` & Matriz de Confusão**: Métricas completas (*Accuracy*, *Macro F1*, *Precision*, *Recall*).
9. **Motor de Inferência de Produção com `pipeline()`**: Construção de um classificador pronto para deploy industrial usando `pipeline("image-classification")`.
10. **Explicabilidade & Interpretabilidade (Attention Rollout)**: Extração dos tensores de atenção com `output_attentions=True` da Hugging Face e visualização de onde o ViT "olhou" na folha ao diagnosticar a patologia.
11. **Desafios Práticos de Engenharia**: Exercícios aplicados de exploração para pós-graduandos.

### 1. Setup do Ambiente, Aceleração de Hardware e Reprodutibilidade

Instalamos as dependências necessárias do ecossistema Hugging Face e PyTorch. No Google Colab, a aceleração via GPU (T4, L4 ou A100) reduz o tempo de treinamento de dezenas de minutos para menos de 2 minutos.

In [ ]:
# Instalação das bibliotecas no Google Colab
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn tqdm pillow

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from transformers import (
    pipeline,
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
)
import datasets
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

# 1. Configuração de Sementes Determinísticas
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# 2. Diagnóstico de Dispositivo e Memória GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_id = 0 if torch.cuda.is_available() else -1
print(f"🔥 Dispositivo de Execução: {device} (Pipeline device: {device_id})")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🎮 GPU Detectada: {gpu_name} ({vram_gb:.2f} GB VRAM)")
else:
    print("⚠️ GPU não detectada. Executando em CPU (recomendado ativar GPU T4 no menu: Ambiente de Execução > Alterar tipo de ambiente de execução).")

### 2. Situação-Problema do Mundo Real: Diagnóstico Agritech em Lavouras de Feijão

Na agricultura de precisão, a rápida identificação de fitopatologias foliares é vital para conter pragas antes que comprometam safras inteiras.

Utilizamos o dataset oficial **`AI-Lab-Makerere/beans`** desenvolvido pelo Laboratório de IA da Makerere University (Uganda). O conjunto consiste em fotos RGB reais capturadas em campo, divididas em 3 classes:
1. **`angular_leaf_spot` (0)**: Mancha Angular (lesões necróticas castanhas delimitadas pelas nervuras).
2. **`bean_rust` (1)**: Ferrugem do Feijoeiro (pequenas pústulas circulares com esporos castanho-avermelhados).
3. **`healthy` (2)**: Folha Saudável (tecido foliar íntegro e homogêneo).

**A Dor de Engenharia:** O dataset possui apenas **1.034 imagens de treino**! Conforme estudamos nos Slides 11 e 12, este é o cenário de maior perigo para o Vision Transformer: devido à ausência de viés indutivo de localidade (diferente das CNNs), treinar um ViT do zero nesse volume de dados levaria a um colapso por memorização superficial (*overfitting*). Por isso, a combinação de **Transfer Learning** a partir de um backbone robusto e **Regularização de Dados** é a solução de engenharia obrigatória.

In [ ]:
# Carregamento do dataset via Hugging Face Datasets
print("📦 Baixando e carregando o dataset AI-Lab-Makerere/beans...")
raw_dataset = datasets.load_dataset("AI-Lab-Makerere/beans")
print(raw_dataset)

# Dicionários de mapeamento entre índices numéricos e classes semânticas
class_names = raw_dataset["train"].features["labels"].names
id2label = {i: name for i, name in enumerate(class_names)}
label2id = {name: i for i, name in enumerate(class_names)}

print("\n🏷️ Classes do Dataset:")
for idx, label in id2label.items():
    print(f"   [{idx}] ➔ {label}")

print(f"\n📊 Distribuição dos Splits:")
print(f"   • Treino:     {len(raw_dataset['train'])} amostras")
print(f"   • Validação:  {len(raw_dataset['validation'])} amostras")
print(f"   • Teste:      {len(raw_dataset['test'])} amostras")

Visualizamos uma amostra representativa de cada uma das 3 condições para inspecionar os traços patológicos que o Vision Transformer precisará aprender a identificar.

In [ ]:
# Visualização exploratória de amostras de cada classe
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, label_name in enumerate(class_names):
    # Encontra a primeira amostra correspondente à classe
    sample = next(item for item in raw_dataset["train"] if item["labels"] == i)
    img = sample["image"]
    
    axes[i].imshow(img)
    axes[i].set_title(f"Classe: {label_name}\nResolução Original: {img.size}", fontsize=12, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Amostras do Dataset de Feijoeiro (AI-Lab Makerere)", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()

### 3. Inferência Zero-Shot de Alto Nível com `transformers.pipeline`

A biblioteca `transformers` oferece uma API de altíssimo nível chamada **`pipeline()`**, que abstrai todo o pré-processamento de imagens, inferência em lote e pós-processamento com apenas uma linha de código.

Vamos utilizar o pipeline oficial `image-classification` com o modelo fundacional **`google/vit-base-patch16-224`** (pré-treinado no ImageNet-1k com 1.000 categorias) diretamente sobre uma folha de feijoeiro com a doença *Bean Rust* (ferrugem).

In [ ]:
MODEL_NAME = "google/vit-base-patch16-224"

# Criação do pipeline de classificação de imagem da Hugging Face
print(f"⚡ Inicializando transformers.pipeline com '{MODEL_NAME}'...")
zero_shot_classifier = pipeline(
    task="image-classification",
    model=MODEL_NAME,
    device=device_id
)

# Amostra de teste com ferrugem (Bean Rust)
test_sample = raw_dataset["test"][5]
test_image = test_sample["image"]
true_label_idx = test_sample["labels"]
true_label_name = id2label[true_label_idx]

# Execução da inferência direta via pipeline da Hugging Face
zero_shot_results = zero_shot_classifier(test_image, top_k=5)

print(f"🎯 Rótulo Real da Amostra: '{true_label_name}'")
print("\n🔮 Top-5 Predições do ViT Original (ImageNet-1k - Sem Fine-Tuning):")
for rank, pred in enumerate(zero_shot_results, start=1):
    print(f"   {rank}. {pred['label']:<32} Confiança: {pred['score']*100:.2f}%")

# Plot da imagem e das predições do pipeline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.imshow(test_image)
ax1.set_title(f"Amostra de Teste\nRótulo Real: {true_label_name}", fontweight="bold")
ax1.axis("off")

labels = [p["label"] for p in zero_shot_results]
scores = [p["score"] * 100 for p in zero_shot_results]
y_pos = np.arange(len(labels))

ax2.barh(y_pos, scores, color="#0A345D")
ax2.set_yticks(y_pos)
ax2.set_yticklabels(labels, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel("Confiança do Pipeline (%)")
ax2.set_title("Inferência Zero-Shot (ImageNet-1k)")
ax2.grid(axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()

**Diagnóstico da Inferência Zero-Shot:**
O pipeline da Hugging Face executou a inferência perfeitamente, mas o modelo pré-treinado no ImageNet prevê rótulos genéricos ("buckeye", "cucumber", "leaf beetle"), pois seu vocabulário de saída desconhece fitopatologias agrícolas.

Isso comprova a necessidade de realizar **Fine-Tuning da Cabeça Linear de Classificação** para o nosso conjunto de dados específico!

### 4. Anatomia Tensorial do Vision Transformer na Hugging Face

Conectando com os **Slides 6 a 10** da aula teórica, inspecionamos como o modelo `ViTForImageClassification` da Hugging Face estrutura os componentes:
- **`model.vit.embeddings.patch_embeddings`**: Camada convolucional `Conv2d(3, 768, kernel_size=16, stride=16)` que fatia a imagem em $14 \times 14 = 196$ patches e projeta para dimensão $768$.
- **`model.vit.embeddings.cls_token`**: Tensor sentinela learnable `[1, 1, 768]`.
- **`model.vit.embeddings.position_embeddings`**: Tensor 1D aprendível `[1, 197, 768]`.
- **`model.vit.encoder.layer`**: 12 blocos Transformer Encoder com Pre-LN, MSA (12 cabeças) e MLP.

In [ ]:
# Carregamento da arquitetura via AutoModel / ViTForImageClassification
base_vit = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    attn_implementation="eager"
).to(device)

total_params = sum(p.numel() for p in base_vit.parameters())
trainable_params = sum(p.numel() for p in base_vit.parameters() if p.requires_grad)

print(f"🧠 Macro-Arquitetura ViT da Hugging Face:")
print(f"   • Total de Parâmetros: {total_params / 1e6:.2f} M")
print(f"   • Projeção Linear de Patches: {base_vit.vit.embeddings.patch_embeddings.projection}")
print(f"   • Tensor do Token [CLS]:      {base_vit.vit.embeddings.cls_token.shape}")
print(f"   • Position Embeddings:         {base_vit.vit.embeddings.position_embeddings.shape}")
print(f"   • Quantidade de Blocos Encoder: {len(base_vit.vit.encoder.layer)}")
print(f"   • Dimensão Oculta (d_model):   {base_vit.config.hidden_size}")
print(f"   • Cabeças de Atenção:          {base_vit.config.num_attention_heads} (d_k = {base_vit.config.hidden_size // base_vit.config.num_attention_heads})")

### 5. Regularização com CutMix: Mitigando o Data Hunger

No **Slide 12**, vimos que o CutMix (Yun et al., ICCV 2019) recorta uma janela de uma imagem e a sobrepõe a outra, ajustando proporcionalmente os rótulos pela área $\lambda$:
$$\tilde{\mathbf{x}} = \mathbf{M} \odot \mathbf{x}_A + (\mathbf{1} - \mathbf{M}) \odot \mathbf{x}_B$$
$$\tilde{y} = \lambda y_A + (1 - \lambda) y_B, \quad \lambda \sim \text{Beta}(\alpha, \alpha)$$

Diferente do Cutout (que gera buracos pretos artificiais) e do Mixup (que cria imagens fantasma transparentes), o CutMix mantém todos os patches 100% naturais. Demonstramos visualmente como essa técnica gera novos exemplos de treino.

In [ ]:
def apply_cutmix_pil(img_a, img_b, alpha=1.0):
    """Aplica CutMix em duas imagens PIL, retornando a imagem combinada e a proporção de área lambda."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    w, h = img_a.size
    
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(w * cut_rat)
    cut_h = int(h * cut_rat)
    
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, w)
    bby1 = np.clip(cy - cut_h // 2, 0, h)
    bbx2 = np.clip(cx + cut_w // 2, 0, w)
    bby2 = np.clip(cy + cut_h // 2, 0, h)
    
    # Recorta região de B e cola sobre A
    combined = img_a.copy()
    crop_b = img_b.crop((bbx1, bby1, bbx2, bby2))
    combined.paste(crop_b, (bbx1, bby1))
    
    actual_lam = 1.0 - ((bbx2 - bbx1) * (bby2 - bby1) / (w * h))
    return combined, actual_lam

# Demonstração do CutMix em pares de folhas de classes distintas
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

samples = [raw_dataset["train"][i] for i in [0, 400, 800, 1000]]
for i in range(4):
    img_a = samples[i]["image"].convert("RGB").resize((224, 224))
    img_b = samples[(i + 1) % 4]["image"].convert("RGB").resize((224, 224))
    
    mixed_img, lam = apply_cutmix_pil(img_a, img_b, alpha=1.0)
    label_a = id2label[samples[i]["labels"]]
    label_b = id2label[samples[(i + 1) % 4]["labels"]]
    
    axes[i].imshow(mixed_img)
    axes[i].set_title(f"CutMix (λ={lam:.2f})\n{lam*100:.0f}% {label_a}\n{(1-lam)*100:.0f}% {label_b}", fontsize=10)
    axes[i].axis("off")

plt.suptitle("Amostras Geradas via CutMix (Patches Naturais e Rótulos Suaves)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

### 6. Pipeline de Pré-Processamento com `ViTImageProcessor` e `datasets`

Utilizamos o **`ViTImageProcessor`** oficial da Hugging Face para redimensionar as imagens para $224 \times 224$ e normalizar com a média e desvio padrão do ImageNet. Aplicamos a transformação funcional sob demanda através de **`dataset.with_transform()`**.

In [ ]:
# Processador oficial do ViT
processor = ViTImageProcessor.from_pretrained(MODEL_NAME)

def transform_batch(example_batch):
    """Aplica o ViTImageProcessor no lote de imagens PIL da Hugging Face."""
    inputs = processor([x.convert("RGB") for x in example_batch["image"]], return_tensors="pt")
    inputs["labels"] = example_batch["labels"]
    return inputs

# Aplicação dinâmica das transformações
prepared_dataset = raw_dataset.with_transform(transform_batch)

def data_collator(batch):
    """Collate function para empilhar tensores de imagem e rótulos."""
    return {
        "pixel_values": torch.stack([x["pixel_values"].squeeze(0) for x in batch]),
        "labels": torch.tensor([x["labels"] for x in batch], dtype=torch.long)
    }

print("✅ Dataset preparado com transformações funcionais da Hugging Face!")

Definimos a função de avaliação `compute_metrics` utilizando a biblioteca **`evaluate`** da Hugging Face para calcular **Acurácia** e **Macro F1-Score** a cada época.

In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    """Calcula métricas de acurácia e macro F1 para o Hugging Face Trainer."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

### 7. Fine-Tuning do ViT com `Trainer` e `TrainingArguments`

Agora realizamos o treinamento completo utilizando o **`Trainer`** da Hugging Face:
- Instanciamos o `ViTForImageClassification` adaptado para as **3 classes** do feijoeiro (`num_labels=3`, `id2label`, `label2id`).
- Configuramos os hiperparâmetros de engenharia via `TrainingArguments`:
  - `learning_rate = 3e-5` com agendador cossenoidal e warmup.
  - `weight_decay = 0.01` (regularização $L_2$).
  - `load_best_model_at_end = True` (salvando o melhor checkpoint baseado na acurácia de validação).

In [ ]:
OUTPUT_DIR = "./vit-beans-finetuned"

# Modelo adaptado com nova cabeça linear de 3 classes
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Substitui a cabeça de 1000 classes por nn.Linear(768, 3)
    attn_implementation="eager"    # Permite inspecionar atenções para explicabilidade visual
).to(device)

print(f"🔍 Nova Cabeça de Classificação Adaptada:\n{model.classifier}")

# Configuração dos argumentos de treino da Hugging Face
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=15,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    remove_unused_columns=False
)

# Instanciação do Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Execução do Fine-Tuning
print("🚀 Iniciando Fine-Tuning do Vision Transformer com o Hugging Face Trainer...")
train_result = trainer.train()

# Salvando o modelo e o processador finais para reutilização imediata no pipeline
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"💾 Modelo e processador salvos com sucesso em: {OUTPUT_DIR}")

### 8. Avaliação no Conjunto de Teste com `Trainer.predict()`

Avaliamos o modelo ajustado sobre o conjunto de teste de 128 amostras independentes utilizando o método nativo `trainer.predict()`. Construímos o relatório de classificação e a matriz de confusão com Seaborn.

In [ ]:
# Avaliação no split de teste
print("📊 Avaliando no split de teste independente...")
test_results = trainer.predict(prepared_dataset["test"])

test_metrics = test_results.metrics
print(f"\n🎯 Métricas Globais no Conjunto de Teste:")
print(f"   • Test Accuracy:  {test_metrics['test_accuracy'] * 100:.2f}%")
print(f"   • Test Macro F1:  {test_metrics['test_f1_macro'] * 100:.2f}%\n")

# Extração de previsões e rótulos
y_pred = np.argmax(test_results.predictions, axis=-1)
y_true = test_results.label_ids

print("📋 Relatório Detalhado por Doença:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Matriz de Confusão no Teste (ViT Fine-Tuned)", fontweight="bold")
plt.xlabel("Classe Prevista pelo Modelo")
plt.ylabel("Classe Real da Folha")
plt.tight_layout()
plt.show()

### 9. Motor de Inferência de Produção com `transformers.pipeline`

Após o fine-tuning, o modelo agora pode ser carregado com o **`pipeline()`** oficial da Hugging Face para realizar inferências de produção em imagens avulsas ou lotes (*batches*) com máxima simplicidade.

Carregamos o modelo salvo em `./vit-beans-finetuned` diretamente no pipeline:

In [ ]:
# Instanciação do Pipeline com o modelo treinado
print(f"⚡ Carregando o pipeline de produção a partir de '{OUTPUT_DIR}'...")
finetuned_classifier = pipeline(
    task="image-classification",
    model=OUTPUT_DIR,
    device=device_id
)

# Teste com uma folha do conjunto de teste
demo_sample = raw_dataset["test"][12]
demo_image = demo_sample["image"]
demo_label = id2label[demo_sample["labels"]]

# Inferência com 1 linha de código via Hugging Face Pipeline
predictions = finetuned_classifier(demo_image, top_k=3)

print(f"🎯 Rótulo Real da Amostra: '{demo_label}'")
print("\n🔮 Diagnóstico do Pipeline Fine-Tuned:")
for pred in predictions:
    print(f"   • {pred['label']:<20} Probabilidade: {pred['score']*100:.2f}%")

# Função de visualização de inferência de produção
def show_pipeline_inference(image, predictions, true_label):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    ax1.imshow(image)
    ax1.set_title(f"Folha Avaliada\nClasse Real: {true_label}", fontweight="bold")
    ax1.axis("off")
    
    labels = [p["label"] for p in predictions]
    scores = [p["score"] * 100 for p in predictions]
    colors = ["#1BB5D8" if l == predictions[0]["label"] else "#94A3B8" for l in labels]
    
    y_pos = np.arange(len(labels))
    bars = ax2.barh(y_pos, scores, color=colors)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(labels, fontsize=11)
    ax2.invert_yaxis()
    ax2.set_xlim(0, 105)
    ax2.set_xlabel("Confiança do Pipeline (%)", fontweight="bold")
    ax2.set_title("Vetor de Probabilidades (Hugging Face)", fontweight="bold")
    ax2.grid(axis="x", linestyle="--", alpha=0.5)
    
    for bar, s in zip(bars, scores):
        ax2.text(s + 1.5, bar.get_y() + bar.get_height()/2, f"{s:.1f}%", va="center", fontweight="bold")
        
    plt.tight_layout()
    plt.show()

show_pipeline_inference(demo_image, predictions, demo_label)

### 10. Explicabilidade & Interpretabilidade: Mapas de Atenção Rollout

No **Slide 13** (*Interpretabilidade: Mapas de Atenção do ViT*), estudamos que cada camada do Vision Transformer calcula uma matriz explícita de autoatenção entre todos os pares de patches.

A biblioteca `transformers` permite inspecionar essas matrizes ativando `output_attentions=True`.
Implementamos o algoritmo **Attention Rollout (Abnar & Zuidema, 2020)** para rastrear o fluxo de atenção do token `[CLS]` em direção aos 196 patches espaciais da folha ao longo de todas as 12 camadas:
1. Em cada camada $\ell$, calculamos a média das 12 cabeças de atenção.
2. Incorporamos a conexão residual: $A_\ell = \frac{1}{2}(W_\ell + I)$.
3. Multiplicamos sucessivamente as matrizes através das camadas: $R = A_{12} \times A_{11} \times \dots \times A_1$.
4. Extraímos os pesos de atenção do token `[CLS]` para os 196 patches, reorganizamos na grade $14 \times 14$ e interpolamos para $224 \times 224$.

In [ ]:
def compute_attention_rollout(attentions):
    """
    Calcula o mapa de Attention Rollout propagando os pesos de atenção
    por todas as 12 camadas do Transformer com conexões residuais.
    """
    result = torch.eye(attentions[0].size(-1)).to(attentions[0].device)
    
    with torch.no_grad():
        for layer_attn in attentions:
            # Média aritmética das 12 cabeças [197, 197]
            attn_fused = layer_attn.squeeze(0).mean(dim=0)
            
            # Adiciona a identidade (conexão residual) e normaliza
            I = torch.eye(attn_fused.size(-1)).to(attn_fused.device)
            a = (attn_fused + I) / 2.0
            a = a / a.sum(dim=-1, keepdim=True)
            
            # Multiplicação matricial em cascata
            result = torch.matmul(a, result)

    # Extração dos pesos do token [CLS] (índice 0) para os 196 patches espaciais (índices 1:)
    cls_attention = result[0, 1:]
    grid_size = int(np.sqrt(cls_attention.size(-1)))  # 14
    heatmap = cls_attention.reshape(grid_size, grid_size).cpu().numpy()
    
    # Normalização min-max
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    return heatmap

def overlay_attention(pil_image, heatmap, alpha=0.55):
    """Interpola o mapa 14x14 para 224x224 e sobrepõe como máscara térmica na imagem."""
    img_resized = pil_image.convert("RGB").resize((224, 224))
    img_arr = np.array(img_resized) / 255.0
    
    # Interpolação bicúbica
    heatmap_pil = Image.fromarray(np.uint8(255 * heatmap)).resize((224, 224), resample=Image.BICUBIC)
    heatmap_norm = np.array(heatmap_pil) / 255.0
    
    # Colormap Jet
    color_map = plt.cm.jet(heatmap_norm)[:, :, :3]
    blended = (1.0 - alpha) * img_arr + alpha * color_map
    return np.clip(blended, 0, 1), heatmap_norm

Auditamos as decisões do modelo em amostras das 3 classes, visualizando onde o Vision Transformer concentrou sua atenção.

In [ ]:
# Modo de avaliação para extração de atenções
model.eval()

sample_test_ids = [3, 8, 22]  # Amostras das 3 classes no conjunto de teste

fig, axes = plt.subplots(len(sample_test_ids), 3, figsize=(14, 4 * len(sample_test_ids)))

for row, sample_idx in enumerate(sample_test_ids):
    sample = raw_dataset["test"][sample_idx]
    image = sample["image"]
    true_cls = id2label[sample["labels"]]
    
    # Inferência via transformers com output_attentions=True
    inputs = processor(images=image.convert("RGB"), return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)
    
    pred_idx = outputs.logits.argmax(dim=-1).item()
    pred_cls = id2label[pred_idx]
    conf = F.softmax(outputs.logits, dim=-1)[0, pred_idx].item() * 100
    
    # Cálculo do Rollout
    heatmap = compute_attention_rollout(outputs.attentions)
    blended, pure_heatmap = overlay_attention(image, heatmap, alpha=0.55)
    
    # 1. Imagem Original
    axes[row, 0].imshow(image.resize((224, 224)))
    axes[row, 0].set_title(f"Amostra #{sample_idx}\nRótulo Real: {true_cls}", fontsize=11, fontweight="bold")
    axes[row, 0].axis("off")
    
    # 2. Mapa de Calor Puro 14x14
    im = axes[row, 1].imshow(pure_heatmap, cmap="magma")
    axes[row, 1].set_title("Atenção [CLS] ➔ Patches (Rollout)", fontsize=11, fontweight="bold")
    axes[row, 1].axis("off")
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)
    
    # 3. Sobreposição de Explicabilidade
    status = "✅" if pred_idx == sample["labels"] else "❌"
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title(f"{status} Previsto: {pred_cls}\nConfiança: {conf:.1f}%", fontsize=11, fontweight="bold")
    axes[row, 2].axis("off")

plt.suptitle("Auditoria Visual via Attention Rollout no Vision Transformer", fontsize=14, fontweight="bold", y=0.99)
plt.tight_layout()
plt.show()

### 11. Desafios Práticos de Engenharia

Para consolidar as competências desta aula, explore os seguintes desafios de engenharia:

1. **Desafio 1 — Feature Extraction vs Fine-Tuning Completo no `Trainer`**:
   - Congele o backbone do Transformer (`for p in model.vit.parameters(): p.requires_grad = False`) e treine apenas o classificador (`model.classifier`).
   - Compare a velocidade de treinamento e a acurácia final em relação ao fine-tuning completo de todas as 12 camadas.
2. **Desafio 2 — O Trade-off do Tamanho do Patch ($16 \times 16$ vs $32 \times 32$)**:
   - Carregue o modelo `google/vit-base-patch32-224` via `pipeline()` e `ViTForImageClassification`.
   - Conforme discutido no **Slide 14**, o patch 32 gera apenas $7 \times 7 = 49$ tokens espaciais (4x menos que os 196 do patch 16).
   - Compare o tempo de inferência por batch e o impacto na acurácia diagnóstica.
3. **Desafio 3 — Comparação com o Swin Transformer**:
   - Utilize a classe `AutoModelForImageClassification` da Hugging Face com o modelo `microsoft/swin-tiny-patch4-window7-224` no mesmo pipeline do `Trainer`.
   - Avalie se as janelas de atenção hierárquicas $O(N)$ do Swin oferecem convergência mais rápida em datasets de tamanho moderado.

---
**Faculdade Infnet — Visão Computacional com CNNs e Transformers**